In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9)

True
NVIDIA L4
23.65915136


```bash
pip install --upgrade pip -q
pip install --upgrade transformers accelerate -q
```

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from transformers import MusicFlamingoForConditionalGeneration, AutoProcessor
print("Import OK")

Import OK


In [7]:
%%bash
ffmpeg -version
python --version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [5]:
!ls

drive  fadl_shaker  sample_data


In [6]:
import torch
from transformers import MusicFlamingoForConditionalGeneration, AutoProcessor

MODEL_ID = "nvidia/music-flamingo-2601-hf"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = MusicFlamingoForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

print("Model loaded. Device map:", model.hf_device_map)
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

processor_config.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

AttributeError: 'MusicFlamingoForConditionalGeneration' object has no attribute 'hf_device_map'

In [7]:
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Model device: {next(model.parameters()).device}")

VRAM used: 16.54 GB
Model device: cuda:0


In [ ]:
import time

AUDIO_PATH = "/content/TRACK_02_01_Rouh.mp3"

MINIMAL_PROMPT = (
    "Listen to this audio track carefully. "
    "List every instrument you can hear. "
    "Then describe the vocal style if vocals are present. "
    "Then estimate the tempo. "
    "Then describe the harmonic or melodic character using interval terms, not genre labels. "
    "Report only what you can directly hear. Do not infer."
)

conversation = [{
    "role": "user",
    "content": [
        {"type": "text", "text": MINIMAL_PROMPT},
        {"type": "audio", "path": AUDIO_PATH},
    ]
}]

start = time.time()

batch = processor.apply_chat_template(
    conversation,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
batch["input_features"] = batch["input_features"].to(model.dtype)

gen_ids = model.generate(**batch, max_new_tokens=256, repetition_penalty=1.2)
inp_len = batch["input_ids"].shape[1]
output = processor.batch_decode(
    gen_ids[:, inp_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

elapsed = time.time() - start
print(f"Time: {elapsed:.1f}s")
print(output)

In [ ]:
import subprocess
result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", "/content/TRACK_02_01_Rouh.mp3"],
    capture_output=True, text=True
)
print(f"Duration: {float(result.stdout.strip()):.1f}s")

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
print(f"Input IDs length: {batch['input_ids'].shape[1]} tokens")
print(f"Input features shape: {batch['input_features'].shape}")

In [ ]:
!python caption.py --file /content/TRACK_02_01_Rouh.mp3

## Minimal

In [ ]:
!python caption.py --file /content/TRACK_02_01_Rouh.mp3 \
  --prompt-mode minimal \
  --output-dir ./captions/phase3_minimal/

## Default

In [ ]:
!python caption.py --file /content/TRACK_02_01_Rouh.mp3 \
  --prompt-mode default \
  --output-dir ./captions/phase3_default/

## Custom

In [ ]:
!python caption.py --file /content/TRACK_02_01_Rouh.mp3 \
  --prompt-mode custom \
  --custom-prompt "List every instrument you hear, then estimate tempo." \
  --output-dir ./captions/phase3_custom/

In [ ]:
%%bash
mkdir -p /content/tracks
cp /content/TRACK_02_01_Rouh.mp3 /content/tracks/
# optionally add a junk file to verify skip behaviour:
touch /content/tracks/notes.txt
touch /content/tracks/.DS_Store

# python caption.py --dir /content/tracks/ # I will upload another mp3 file too (me the user)
ls ./tracks/


In [ ]:
!python caption.py --dir /content/tracks/ # I will upload another mp3 file too (me the user)

In [ ]:
!pytest test_caption.py -v

In [ ]:
ls ./FADL_SHAKER/

# Layer 2 Testing

In [3]:
!python caption.py --job fadl_shaker_512.json

Loading processor from 'nvidia/music-flamingo-2601-hf' ...
Loading model (precision=bf16) ...
Loading weights: 100% 830/830 [00:04<00:00, 195.81it/s]
Model ready. Device: cuda:0 | VRAM allocated: 16.54 GB
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1200). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[1/4] 03.Allah We'allam.mp3 → OK (21.6s)
[2/4] 07.M'assar Feya.mp3 → OK (33.5s)
[3/4] 12.Dehket El Donya.mp3 → OK (19.8s)
[4/4] 16.Baya' El Oulob.mp3 → OK (15.6s)

Processed: 4 | OK: 4 | Failed: 0 | Summary → ./captions/fadl_shaker_512/results_summary.json


In [4]:
!python caption.py --job fadl_shaker_acoustic_only.json

Loading processor from 'nvidia/music-flamingo-2601-hf' ...
Loading model (precision=bf16) ...
Loading weights: 100% 830/830 [00:04<00:00, 193.78it/s]
Model ready. Device: cuda:0 | VRAM allocated: 16.54 GB
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1200). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[1/4] 03.Allah We'allam.mp3 → OK (14.0s)
[2/4] 07.M'assar Feya.mp3 → OK (14.2s)
[3/4] 12.Dehket El Donya.mp3 → OK (13.9s)
[4/4] 16.Baya' El Oulob.mp3 → OK (13.8s)

Processed: 4 | OK: 4 | Failed: 0 | Summary → ./captions/fadl_shaker_acoustic_only/results_summary.json


In [5]:
!python caption.py --job fadl_shaker_prose_prompt.json

Loading processor from 'nvidia/music-flamingo-2601-hf' ...
Loading model (precision=bf16) ...
Loading weights: 100% 830/830 [00:04<00:00, 196.52it/s]
Model ready. Device: cuda:0 | VRAM allocated: 16.54 GB
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1200). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[1/4] 03.Allah We'allam.mp3 → OK (20.2s)
[2/4] 07.M'assar Feya.mp3 → OK (19.8s)
[3/4] 12.Dehket El Donya.mp3 → OK (19.3s)
[4/4] 16.Baya' El Oulob.mp3 → OK (19.9s)

Processed: 4 | OK: 4 | Failed: 0 | Summary → ./captions/fadl_shaker_prose_prompt/results_summary.json


In [6]:
!python ./xml_directory_processor.py ./captions/fadl_shaker_prose_prompt/ --o ./fadel_shaker_prose.txt


Processing complete!
Total tokens: 2,622
Output written to: ./fadel_shaker_prose.txt


In [7]:
!python caption.py --job fadl_shaker_hybrid.json

Loading processor from 'nvidia/music-flamingo-2601-hf' ...
Loading model (precision=bf16) ...
Loading weights: 100% 830/830 [00:04<00:00, 196.90it/s]
Model ready. Device: cuda:0 | VRAM allocated: 16.54 GB
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1200). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[1/4] 03.Allah We'allam.mp3 → OK (20.2s)
[2/4] 07.M'assar Feya.mp3 → OK (19.8s)
[3/4] 12.Dehket El Donya.mp3 → OK (19.4s)
[4/4] 16.Baya' El Oulob.mp3 → OK (19.9s)

Processed: 4 | OK: 4 | Failed: 0 | Summary → ./captions/fadl_shaker_hybrid/results_summary.json


In [8]:
!python ./xml_directory_processor.py ./captions/fadl_shaker_hybrid/ --o ./fadel_shaker_hybrid.txt


Processing complete!
Total tokens: 2,753
Output written to: ./fadel_shaker_hybrid.txt


In [9]:
!python ./xml_directory_processor.py ./captions/ --o ./fadel_shaker_ALL.txt


Processing complete!
Total tokens: 10,889
Output written to: ./fadel_shaker_ALL.txt
